# Phase2 EQ Emotion Experiment Analysis

This notebook analyzes whether emotion-model output aligns with questionnaire **Q1**.

You only need to modify **three objects** in the next cell:

1. `emotion_data` (list per subject, each listening has 4 raw + 4 vote)
2. `questionnaire` (list per subject, min 5 and max 20 entries)
3. `ranking` (EQ1~EQ5 ranking list per subject)

You can also tune vote analysis parameters:

- vote label weights (`positive`, `neutral`, `negative`)
- vote window range (`window_start`, `window_end`)

Outputs include:

- Per-subject first-round Q1/Q2 (first exposure of each EQ)
- Per-subject all-round mean Q1/Q2
- Group first-round Q1 mean and Q1 baseline effect (EQ3)
- Q1-based rank tables (Q1 mean rank, Q1 effect rank)
- Emotion model vote/raw score/rank tables
- Grid search table to find vote settings closest to survey Q1 ranking


In [9]:
# =============================
# INPUT DATA (EDIT THESE ONLY)
# =============================

EQ_LIST = [f"EQ{i}" for i in range(1, 6)]

# Rebuilt-vote analysis config (tunable)
# Use slice [window_start:window_end) on each session's 4 RAW windows.
# New vote score is computed from RAW labels in this window using weights below.
# Example: (3,4)=only 4th raw window, (2,4)=last two windows, (0,4)=all windows.
VOTE_ANALYSIS_CONFIG = {
    "weights": {
        "positive": -1.0,
        "neutral":  0.5,
        "negative": 0.0
    },
    "window_start": 0,
    "window_end": 4
}

# Emotion model output
# Format: subject -> list of listening sessions
# Each session contains 4 raw labels and 4 vote labels (time is optional metadata)
# source_eq keeps original EQ id from csv (E1/E2/E7/E15/E16).
# Mapping used here: E1->EQ1, E2->EQ2, E7->EQ3, E15->EQ4, E16->EQ5
emotion_data = {'david': [{'eq': 'EQ1',
            'source_eq': 'E1',
            'time': ['2026-03-12 10:44:02.783 ~ 2026-03-12 10:44:17.280',
                     '2026-03-12 10:44:17.280 ~ 2026-03-12 10:44:33.260',
                     '2026-03-12 10:44:33.260 ~ 2026-03-12 10:44:48.413',
                     '2026-03-12 10:44:48.413 ~ 2026-03-12 10:45:04.275'],
            'raw': ['positive', 'neutral', 'neutral', 'neutral'],
            'vote': ['positive', 'positive', 'negative', 'negative']},
           {'eq': 'EQ2',
            'source_eq': 'E2',
            'time': ['2026-03-12 10:45:22.782 ~ 2026-03-12 10:45:37.243',
                     '2026-03-12 10:45:37.243 ~ 2026-03-12 10:45:53.335',
                     '2026-03-12 10:45:53.335 ~ 2026-03-12 10:46:08.314',
                     '2026-03-12 10:46:08.314 ~ 2026-03-12 10:46:24.351'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ3',
            'source_eq': 'E7',
            'time': ['2026-03-12 10:46:39.783 ~ 2026-03-12 10:46:54.457',
                     '2026-03-12 10:46:54.457 ~ 2026-03-12 10:47:10.335',
                     '2026-03-12 10:47:10.335 ~ 2026-03-12 10:47:25.294',
                     '2026-03-12 10:47:25.294 ~ 2026-03-12 10:47:41.358'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ4',
            'source_eq': 'E15',
            'time': ['2026-03-12 10:47:50.783 ~ 2026-03-12 10:48:05.317',
                     '2026-03-12 10:48:05.317 ~ 2026-03-12 10:48:21.295',
                     '2026-03-12 10:48:21.295 ~ 2026-03-12 10:48:36.351',
                     '2026-03-12 10:48:36.351 ~ 2026-03-12 10:48:51.297'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ5',
            'source_eq': 'E16',
            'time': ['2026-03-12 10:48:57.783 ~ 2026-03-12 10:49:13.312',
                     '2026-03-12 10:49:13.312 ~ 2026-03-12 10:49:28.265',
                     '2026-03-12 10:49:28.265 ~ 2026-03-12 10:49:44.340',
                     '2026-03-12 10:49:44.340 ~ 2026-03-12 10:49:59.295'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ1',
            'source_eq': 'E1',
            'time': ['2026-03-12 10:50:04.789 ~ 2026-03-12 10:50:20.282',
                     '2026-03-12 10:50:20.282 ~ 2026-03-12 10:50:35.334',
                     '2026-03-12 10:50:35.334 ~ 2026-03-12 10:50:50.287',
                     '2026-03-12 10:50:50.287 ~ 2026-03-12 10:51:06.363'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ2',
            'source_eq': 'E2',
            'time': ['2026-03-12 10:51:22.782 ~ 2026-03-12 10:51:38.313',
                     '2026-03-12 10:51:38.313 ~ 2026-03-12 10:51:53.265',
                     '2026-03-12 10:51:53.265 ~ 2026-03-12 10:52:09.342',
                     '2026-03-12 10:52:09.342 ~ 2026-03-12 10:52:24.294'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ3',
            'source_eq': 'E7',
            'time': ['2026-03-12 10:52:27.782 ~ 2026-03-12 10:52:42.310',
                     '2026-03-12 10:52:42.310 ~ 2026-03-12 10:52:58.321',
                     '2026-03-12 10:52:58.321 ~ 2026-03-12 10:53:13.351',
                     '2026-03-12 10:53:13.351 ~ 2026-03-12 10:53:28.289'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ4',
            'source_eq': 'E15',
            'time': ['2026-03-12 10:53:39.782 ~ 2026-03-12 10:53:54.302',
                     '2026-03-12 10:53:54.302 ~ 2026-03-12 10:54:09.288',
                     '2026-03-12 10:54:09.288 ~ 2026-03-12 10:54:25.332',
                     '2026-03-12 10:54:25.332 ~ 2026-03-12 10:54:40.277'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ5',
            'source_eq': 'E16',
            'time': ['2026-03-12 10:54:43.784 ~ 2026-03-12 10:54:59.332',
                     '2026-03-12 10:54:59.332 ~ 2026-03-12 10:55:14.297',
                     '2026-03-12 10:55:14.297 ~ 2026-03-12 10:55:29.328',
                     '2026-03-12 10:55:29.328 ~ 2026-03-12 10:55:46.328'],
            'raw': ['neutral', 'positive', 'neutral', 'neutral'],
            'vote': ['negative', 'positive', 'negative', 'negative']}],
 'ken': [{'eq': 'EQ1',
          'source_eq': 'E1',
          'time': ['2026-03-12 09:23:45.642 ~ 2026-03-12 09:24:00.111',
                   '2026-03-12 09:24:00.111 ~ 2026-03-12 09:24:15.174',
                   '2026-03-12 09:24:15.174 ~ 2026-03-12 09:24:31.141',
                   '2026-03-12 09:24:31.141 ~ 2026-03-12 09:24:46.197'],
          'raw': ['positive', 'positive', 'neutral', 'positive'],
          'vote': ['positive', 'positive', 'positive', 'positive']},
         {'eq': 'EQ2',
          'source_eq': 'E2',
          'time': ['2026-03-12 09:24:54.642 ~ 2026-03-12 09:25:11.182',
                   '2026-03-12 09:25:11.182 ~ 2026-03-12 09:25:26.128',
                   '2026-03-12 09:25:26.128 ~ 2026-03-12 09:25:41.182',
                   '2026-03-12 09:25:41.182 ~ 2026-03-12 09:25:57.168'],
          'raw': ['neutral', 'positive', 'positive', 'positive'],
          'vote': ['negative', 'positive', 'positive', 'positive']},
         {'eq': 'EQ3',
          'source_eq': 'E7',
          'time': ['2026-03-12 09:26:02.642 ~ 2026-03-12 09:26:17.127',
                   '2026-03-12 09:26:17.127 ~ 2026-03-12 09:26:34.124',
                   '2026-03-12 09:26:34.124 ~ 2026-03-12 09:26:49.182',
                   '2026-03-12 09:26:49.182 ~ 2026-03-12 09:27:04.095'],
          'raw': ['neutral', 'positive', 'positive', 'neutral'],
          'vote': ['negative', 'positive', 'positive', 'positive']},
         {'eq': 'EQ4',
          'source_eq': 'E15',
          'time': ['2026-03-12 09:27:12.642 ~ 2026-03-12 09:27:27.182',
                   '2026-03-12 09:27:27.182 ~ 2026-03-12 09:27:42.120',
                   '2026-03-12 09:27:42.120 ~ 2026-03-12 09:27:58.513',
                   '2026-03-12 09:27:58.513 ~ 2026-03-12 09:28:13.158'],
          'raw': ['neutral', 'positive', 'positive', 'neutral'],
          'vote': ['negative', 'positive', 'positive', 'positive']},
         {'eq': 'EQ5',
          'source_eq': 'E16',
          'time': ['2026-03-12 09:28:20.642 ~ 2026-03-12 09:28:37.126',
                   '2026-03-12 09:28:37.126 ~ 2026-03-12 09:28:52.263',
                   '2026-03-12 09:28:52.263 ~ 2026-03-12 09:29:07.117',
                   '2026-03-12 09:29:07.117 ~ 2026-03-12 09:29:24.109'],
          'raw': ['neutral', 'neutral', 'positive', 'positive'],
          'vote': ['negative', 'negative', 'negative', 'positive']},
         {'eq': 'EQ2',
          'source_eq': 'E2',
          'time': ['2026-03-12 09:29:27.642 ~ 2026-03-12 09:29:42.136',
                   '2026-03-12 09:29:42.136 ~ 2026-03-12 09:29:57.186',
                   '2026-03-12 09:29:57.186 ~ 2026-03-12 09:30:13.168',
                   '2026-03-12 09:30:13.168 ~ 2026-03-12 09:30:28.073'],
          'raw': ['positive', 'neutral', 'neutral', 'neutral'],
          'vote': ['positive', 'positive', 'negative', 'negative']}],
 'parker': [{'eq': 'EQ1',
             'source_eq': 'E1',
             'time': ['2026-03-12 09:43:31.329 ~ 2026-03-12 09:43:45.925',
                      '2026-03-12 09:43:45.925 ~ 2026-03-12 09:44:01.893',
                      '2026-03-12 09:44:01.893 ~ 2026-03-12 09:44:16.842',
                      '2026-03-12 09:44:16.842 ~ 2026-03-12 09:44:32.822'],
             'raw': ['positive', 'positive', 'positive', 'positive'],
             'vote': ['positive', 'positive', 'positive', 'positive']},
            {'eq': 'EQ2',
             'source_eq': 'E2',
             'time': ['2026-03-12 09:44:39.334 ~ 2026-03-12 09:44:53.807',
                      '2026-03-12 09:44:53.807 ~ 2026-03-12 09:45:09.782',
                      '2026-03-12 09:45:09.782 ~ 2026-03-12 09:45:24.838',
                      'missing_window'],
             'raw': ['neutral', 'positive', 'neutral', 'unknown'],
             'vote': ['negative', 'positive', 'negative', 'negative']},
            {'eq': 'EQ3',
             'source_eq': 'E7',
             'time': ['2026-03-12 09:45:43.327 ~ 2026-03-12 09:45:57.813',
                      '2026-03-12 09:45:57.813 ~ 2026-03-12 09:46:12.866',
                      '2026-03-12 09:46:12.866 ~ 2026-03-12 09:46:28.836',
                      '2026-03-12 09:46:28.836 ~ 2026-03-12 09:46:43.888'],
             'raw': ['neutral', 'positive', 'neutral', 'positive'],
             'vote': ['negative', 'positive', 'negative', 'positive']},
            {'eq': 'EQ4',
             'source_eq': 'E15',
             'time': ['2026-03-12 09:46:48.328 ~ 2026-03-12 09:47:03.858',
                      '2026-03-12 09:47:03.858 ~ 2026-03-12 09:47:18.807',
                      '2026-03-12 09:47:18.807 ~ 2026-03-12 09:47:34.781',
                      '2026-03-12 09:47:34.781 ~ 2026-03-12 09:47:49.936'],
             'raw': ['positive', 'positive', 'neutral', 'positive'],
             'vote': ['positive', 'positive', 'positive', 'positive']},
            {'eq': 'EQ5',
             'source_eq': 'E16',
             'time': ['2026-03-12 09:47:51.328 ~ 2026-03-12 09:48:06.839',
                      '2026-03-12 09:48:06.839 ~ 2026-03-12 09:48:21.897',
                      '2026-03-12 09:48:21.897 ~ 2026-03-12 09:48:36.837',
                      '2026-03-12 09:48:36.837 ~ 2026-03-12 09:48:52.914'],
             'raw': ['neutral', 'neutral', 'neutral', 'positive'],
             'vote': ['negative', 'negative', 'negative', 'negative']},
            {'eq': 'EQ3',
             'source_eq': 'E7',
             'time': ['2026-03-12 09:49:05.327 ~ 2026-03-12 09:49:19.845',
                      '2026-03-12 09:49:19.845 ~ 2026-03-12 09:49:36.842',
                      '2026-03-12 09:49:36.842 ~ 2026-03-12 09:50:07.869',
                      'missing_window'],
             'raw': ['positive', 'positive', 'positive', 'unknown'],
             'vote': ['positive', 'positive', 'positive', 'positive']},
            {'eq': 'EQ1',
             'source_eq': 'E1',
             'time': ['2026-03-12 09:50:10.328 ~ 2026-03-12 09:50:24.973',
                      '2026-03-12 09:50:24.973 ~ 2026-03-12 09:50:40.845',
                      '2026-03-12 09:50:40.845 ~ 2026-03-12 09:51:10.846',
                      'missing_window'],
             'raw': ['neutral', 'positive', 'positive', 'unknown'],
             'vote': ['negative', 'positive', 'positive', 'positive']},
            {'eq': 'EQ2',
             'source_eq': 'E2',
             'time': ['2026-03-12 09:51:11.326 ~ 2026-03-12 09:51:26.819',
                      '2026-03-12 09:51:26.819 ~ 2026-03-12 09:51:41.874',
                      '2026-03-12 09:51:41.874 ~ 2026-03-12 09:51:57.849',
                      '2026-03-12 09:51:57.849 ~ 2026-03-12 09:52:12.800'],
             'raw': ['neutral', 'positive', 'neutral', 'positive'],
             'vote': ['negative', 'positive', 'negative', 'positive']}],
 'roger': [{'eq': 'EQ1',
            'source_eq': 'E1',
            'time': ['2026-03-12 09:56:27.376 ~ 2026-03-12 09:56:42.944',
                     '2026-03-12 09:56:42.944 ~ 2026-03-12 09:56:57.884',
                     '2026-03-12 09:56:57.884 ~ 2026-03-12 09:57:13.862',
                     '2026-03-12 09:57:13.862 ~ 2026-03-12 09:57:28.913'],
            'raw': ['neutral', 'neutral', 'neutral', 'positive'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ2',
            'source_eq': 'E2',
            'time': ['2026-03-12 09:57:42.376 ~ 2026-03-12 09:57:57.901',
                     '2026-03-12 09:57:57.901 ~ 2026-03-12 09:58:12.853',
                     '2026-03-12 09:58:12.853 ~ 2026-03-12 09:58:28.924',
                     '2026-03-12 09:58:28.924 ~ 2026-03-12 09:58:43.876'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ3',
            'source_eq': 'E7',
            'time': ['2026-03-12 09:58:53.380 ~ 2026-03-12 09:59:07.839',
                     '2026-03-12 09:59:07.839 ~ 2026-03-12 09:59:24.840',
                     '2026-03-12 09:59:24.840 ~ 2026-03-12 09:59:39.890',
                     '2026-03-12 09:59:39.890 ~ 2026-03-12 09:59:55.868'],
            'raw': ['positive', 'neutral', 'neutral', 'neutral'],
            'vote': ['positive', 'positive', 'negative', 'negative']},
           {'eq': 'EQ4',
            'source_eq': 'E15',
            'time': ['2026-03-12 10:00:06.376 ~ 2026-03-12 10:00:20.852',
                     '2026-03-12 10:00:20.852 ~ 2026-03-12 10:00:35.904',
                     '2026-03-12 10:00:35.904 ~ 2026-03-12 10:00:51.880',
                     '2026-03-12 10:00:51.880 ~ 2026-03-12 10:01:06.933'],
            'raw': ['neutral', 'positive', 'neutral', 'neutral'],
            'vote': ['negative', 'positive', 'negative', 'negative']},
           {'eq': 'EQ5',
            'source_eq': 'E16',
            'time': ['2026-03-12 10:01:12.375 ~ 2026-03-12 10:01:27.940',
                     '2026-03-12 10:01:27.940 ~ 2026-03-12 10:01:42.885',
                     '2026-03-12 10:01:42.885 ~ 2026-03-12 10:01:58.042',
                     '2026-03-12 10:01:58.042 ~ 2026-03-12 10:02:13.911'],
            'raw': ['positive', 'neutral', 'neutral', 'neutral'],
            'vote': ['positive', 'positive', 'negative', 'negative']},
           {'eq': 'EQ1',
            'source_eq': 'E1',
            'time': ['2026-03-12 10:02:18.376 ~ 2026-03-12 10:02:32.850',
                     '2026-03-12 10:02:32.850 ~ 2026-03-12 10:02:47.906',
                     '2026-03-12 10:02:47.906 ~ 2026-03-12 10:03:03.883',
                     '2026-03-12 10:03:03.883 ~ 2026-03-12 10:03:18.935'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ3',
            'source_eq': 'E7',
            'time': ['2026-03-12 10:03:31.381 ~ 2026-03-12 10:03:46.886',
                     '2026-03-12 10:03:46.886 ~ 2026-03-12 10:04:01.942',
                     '2026-03-12 10:04:01.942 ~ 2026-03-12 10:04:16.890',
                     '2026-03-12 10:04:16.890 ~ 2026-03-12 10:04:33.896'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ2',
            'source_eq': 'E2',
            'time': ['2026-03-12 10:04:37.375 ~ 2026-03-12 10:04:51.912',
                     '2026-03-12 10:04:51.912 ~ 2026-03-12 10:05:06.866',
                     '2026-03-12 10:05:06.866 ~ 2026-03-12 10:05:22.946',
                     '2026-03-12 10:05:22.946 ~ 2026-03-12 10:05:37.895'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ1',
            'source_eq': 'E1',
            'time': ['2026-03-12 10:05:45.375 ~ 2026-03-12 10:05:59.913',
                     '2026-03-12 10:05:59.913 ~ 2026-03-12 10:06:15.895',
                     '2026-03-12 10:06:15.895 ~ 2026-03-12 10:06:30.940',
                     '2026-03-12 10:06:30.940 ~ 2026-03-12 10:06:45.887'],
            'raw': ['neutral', 'neutral', 'positive', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ3',
            'source_eq': 'E7',
            'time': ['2026-03-12 10:06:49.375 ~ 2026-03-12 10:07:03.911',
                     '2026-03-12 10:07:03.911 ~ 2026-03-12 10:07:18.864',
                     '2026-03-12 10:07:18.864 ~ 2026-03-12 10:07:35.861',
                     '2026-03-12 10:07:35.861 ~ 2026-03-12 10:07:50.825'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ2',
            'source_eq': 'E2',
            'time': ['2026-03-12 10:07:51.375 ~ 2026-03-12 10:08:06.891',
                     '2026-03-12 10:08:06.891 ~ 2026-03-12 10:08:21.945',
                     '2026-03-12 10:08:21.945 ~ 2026-03-12 10:08:36.897',
                     '2026-03-12 10:08:36.897 ~ 2026-03-12 10:08:52.973'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ3',
            'source_eq': 'E7',
            'time': ['2026-03-12 10:09:00.375 ~ 2026-03-12 10:09:14.847',
                     '2026-03-12 10:09:14.847 ~ 2026-03-12 10:09:29.935',
                     '2026-03-12 10:09:29.935 ~ 2026-03-12 10:09:44.894',
                     '2026-03-12 10:09:44.894 ~ 2026-03-12 10:10:00.861'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ1',
            'source_eq': 'E1',
            'time': ['2026-03-12 10:10:05.379 ~ 2026-03-12 10:10:19.907',
                     '2026-03-12 10:10:19.907 ~ 2026-03-12 10:10:34.853',
                     '2026-03-12 10:10:34.853 ~ 2026-03-12 10:10:51.038',
                     '2026-03-12 10:10:51.038 ~ 2026-03-12 10:11:05.886'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']},
           {'eq': 'EQ3',
            'source_eq': 'E7',
            'time': ['2026-03-12 10:11:08.375 ~ 2026-03-12 10:11:23.911',
                     '2026-03-12 10:11:23.911 ~ 2026-03-12 10:11:38.859',
                     '2026-03-12 10:11:38.859 ~ 2026-03-12 10:11:53.913',
                     '2026-03-12 10:11:53.913 ~ 2026-03-12 10:12:09.990'],
            'raw': ['neutral', 'positive', 'neutral', 'neutral'],
            'vote': ['negative', 'positive', 'negative', 'negative']},
           {'eq': 'EQ2',
            'source_eq': 'E2',
            'time': ['2026-03-12 10:12:11.376 ~ 2026-03-12 10:12:25.868',
                     '2026-03-12 10:12:25.868 ~ 2026-03-12 10:12:43.075',
                     '2026-03-12 10:12:43.075 ~ 2026-03-12 10:12:57.915',
                     '2026-03-12 10:12:57.915 ~ 2026-03-12 10:13:12.864'],
            'raw': ['neutral', 'neutral', 'neutral', 'neutral'],
            'vote': ['negative', 'negative', 'negative', 'negative']}]}

# Questionnaire data (subjective responses)
# Format: subject -> ordered list of listening entries
# Rule: minimum 5 entries (first pass of EQ1~EQ5), maximum 20 entries
questionnaire = {
    "ken": [
        {"eq": "EQ1", "Q1": 3, "Q2": 3},
        {"eq": "EQ2", "Q1": 3, "Q2": 2},
        {"eq": "EQ3", "Q1": 3, "Q2": 3},
        {"eq": "EQ4", "Q1": 2, "Q2": 4},
        {"eq": "EQ5", "Q1": 1, "Q2": 4},
        {"eq": "EQ2", "Q1": 4, "Q2": 2}
    ],
    "Parker": [
        {"eq": "EQ1", "Q1": 3, "Q2": 3},
        {"eq": "EQ2", "Q1": 4, "Q2": 2},
        {"eq": "EQ3", "Q1": 4, "Q2": 2},
        {"eq": "EQ4", "Q1": 1, "Q2": 4},
        {"eq": "EQ5", "Q1": 2, "Q2": 4},
        {"eq": "EQ1", "Q1": 4, "Q2": 2},
        {"eq": "EQ2", "Q1": 4, "Q2": 2},
        {"eq": "EQ3", "Q1": 4, "Q2": 2}
    ],
    "Roger": [
        {"eq": "EQ1", "Q1": 3, "Q2": 4},
        {"eq": "EQ2", "Q1": 4, "Q2": 3},
        {"eq": "EQ4", "Q1": 2, "Q2": 4},
        {"eq": "EQ5", "Q1": 2, "Q2": 4},
        {"eq": "EQ1", "Q1": 3, "Q2": 3},
        {"eq": "EQ2", "Q1": 4, "Q2": 2},
        {"eq": "EQ3", "Q1": 3, "Q2": 2},
        {"eq": "EQ1", "Q1": 3, "Q2": 3},
        {"eq": "EQ2", "Q1": 3, "Q2": 3},
        {"eq": "EQ3", "Q1": 4, "Q2": 1},
        {"eq": "EQ1", "Q1": 3, "Q2": 3},
        {"eq": "EQ2", "Q1": 4, "Q2": 3},
        {"eq": "EQ3", "Q1": 4, "Q2": 2},
        {"eq": "EQ3", "Q1": 5, "Q2": 1}
    ],
    "David": [
        {"eq": "EQ1", "Q1": 3, "Q2": 4},
        {"eq": "EQ2", "Q1": 3, "Q2": 3},
        {"eq": "EQ3", "Q1": 4, "Q2": 2},
        {"eq": "EQ4", "Q1": 2, "Q2": 4},
        {"eq": "EQ5", "Q1": 3, "Q2": 3},
        {"eq": "EQ1", "Q1": 4, "Q2": 3},
        {"eq": "EQ2", "Q1": 4, "Q2": 2},
        {"eq": "EQ3", "Q1": 3, "Q2": 3},
        {"eq": "EQ4", "Q1": 2, "Q2": 4},
        {"eq": "EQ5", "Q1": 3, "Q2": 4}
    ]
    
}

# Ranking results (EQ1~EQ5)
# Lower score is better rank (1 = best)
ranking = {
    "ken": [2, 1, 3, 4, 5],
    "Parker": [3, 1, 2, 5, 4],
    "Roger": [3, 2, 1, 5, 4]
}


In [10]:
import pandas as pd
import numpy as np

ALLOWED_VOTE = {"positive", "negative", "neutral"}
ALLOWED_RAW = {"positive", "negative", "neutral", "unknown"}
SCORE_MAP = {"positive": 1, "neutral": 0, "negative": -1}  #維持vote v0的分數


def _rank_desc(series):
    # Higher score gets better rank (1 = best)
    return series.rank(method="min", ascending=False).astype("Int64")


def _rank_asc(series):
    # Lower value gets better rank (1 = best)
    return series.rank(method="min", ascending=True).astype("Int64")


def get_raw_window_slice(config):
    ws = int(config.get("window_start", 0))
    we = int(config.get("window_end", 4))
    if ws < 0 or we > 4 or ws >= we:
        raise ValueError(f"Invalid raw window slice: start={ws}, end={we}. Must satisfy 0 <= start < end <= 4")
    return ws, we


def validate_inputs(eq_list, emotion, questionnaire_data, ranking_data, vote_config):
    errors = []

    try:
        get_raw_window_slice(vote_config)
    except Exception as e:
        errors.append(str(e))

    weights = vote_config.get("weights", {})
    for label in ["positive", "neutral", "negative"]:
        if label not in weights:
            errors.append(f"VOTE_ANALYSIS_CONFIG.weights missing '{label}'")

    for subject, entries in questionnaire_data.items():
        if not (5 <= len(entries) <= 20):
            errors.append(f"{subject}: questionnaire entries must be between 5 and 20 (got {len(entries)})")

        seen_eq = {e.get("eq") for e in entries}
        missing = [eq for eq in eq_list if eq not in seen_eq]
        if missing:
            errors.append(f"{subject}: first-pass requirement missing EQs {missing}")

        for i, e in enumerate(entries, start=1):
            eq = e.get("eq")
            q1 = e.get("Q1")
            q2 = e.get("Q2")
            if eq not in eq_list:
                errors.append(f"{subject}: questionnaire entry {i} has invalid eq '{eq}'")
            if q1 is None or not (1 <= q1 <= 5):
                errors.append(f"{subject}: questionnaire entry {i} has invalid Q1 '{q1}'")
            if q2 is None or not (1 <= q2 <= 5):
                errors.append(f"{subject}: questionnaire entry {i} has invalid Q2 '{q2}'")

    for subject, sessions in emotion.items():
        for i, s in enumerate(sessions, start=1):
            eq = s.get("eq")
            raw = s.get("raw", [])
            vote = s.get("vote", [])
            if eq not in eq_list:
                errors.append(f"{subject}: emotion session {i} has invalid eq '{eq}'")
            if len(raw) != 4:
                errors.append(f"{subject}: emotion session {i} raw length must be 4 (got {len(raw)})")
            if len(vote) != 4:
                errors.append(f"{subject}: emotion session {i} vote length must be 4 (got {len(vote)})")

            invalid_raw = [r for r in raw if r not in ALLOWED_RAW]
            if invalid_raw:
                errors.append(f"{subject}: emotion session {i} has invalid raw labels {invalid_raw}")

            invalid_vote = [v for v in vote if v not in ALLOWED_VOTE]
            if invalid_vote:
                errors.append(f"{subject}: emotion session {i} has invalid vote labels {invalid_vote}")

    for subject, ranks in ranking_data.items():
        if len(ranks) != len(eq_list):
            errors.append(f"{subject}: ranking must have {len(eq_list)} values (got {len(ranks)})")
            continue
        rank_set = set(ranks)
        expected = set(range(1, len(eq_list) + 1))
        if rank_set != expected:
            errors.append(f"{subject}: ranking must be a permutation of 1..{len(eq_list)}")

    if errors:
        raise ValueError("Input validation failed:" + "\n- " + "\n- ".join(errors))


validate_inputs(EQ_LIST, emotion_data, questionnaire, ranking, VOTE_ANALYSIS_CONFIG)


## First Round Q1/Q2 Analysis

In [11]:
# Per-subject first-round Q1/Q2 (first exposure of each EQ)
first_rows = []

for subject, entries in questionnaire.items():
    seen = {}
    for e in entries:
        eq = e["eq"]
        if eq not in seen:
            seen[eq] = e
        if len(seen) == len(EQ_LIST):
            break

    for eq in EQ_LIST:
        entry = seen[eq]
        first_rows.append({
            "Subject": subject,
            "EQ": eq,
            "First_Q1": entry["Q1"],
            "First_Q2": entry["Q2"]
        })

first_subject_eq = pd.DataFrame(first_rows).sort_values(["Subject", "EQ"]).reset_index(drop=True)
first_subject_eq


,Subject,EQ,First_Q1,First_Q2
0,David,EQ1,3,4
1,David,EQ2,3,3
2,David,EQ3,4,2
3,David,EQ4,2,4
4,David,EQ5,3,3
5,Parker,EQ1,3,3
6,Parker,EQ2,4,2
7,Parker,EQ3,4,2
8,Parker,EQ4,1,4
9,Parker,EQ5,2,4


## All-Rounds Mean Q1/Q2

In [12]:
# Per-subject all-round mean Q1/Q2 and group first-round summary
all_rows = []

for subject, entries in questionnaire.items():
    for e in entries:
        all_rows.append({
            "Subject": subject,
            "EQ": e["eq"],
            "Q1": e["Q1"],
            "Q2": e["Q2"]
        })

all_long = pd.DataFrame(all_rows)

all_subject_eq = (
    all_long.groupby(["Subject", "EQ"], as_index=False)
    .agg(
        Mean_Q1=("Q1", "mean"),
        Mean_Q2=("Q2", "mean"),
        Listen_Count=("Q1", "size")
    )
)
all_subject_eq = all_subject_eq.sort_values(["Subject", "EQ"]).reset_index(drop=True)

# Group first-round means across all subjects (Q1 is primary target)
first_group_eq = (
    first_subject_eq.groupby("EQ", as_index=False)
    .agg(
        First_Q1_Mean=("First_Q1", "mean"),
        First_Q2_Mean=("First_Q2", "mean"),
        N_Subjects=("Subject", "nunique")
    )
)
first_group_eq = first_group_eq.sort_values("EQ").reset_index(drop=True)

all_subject_eq


,Subject,EQ,Mean_Q1,Mean_Q2,Listen_Count
0,David,EQ1,3.50,3.50,2
1,David,EQ2,3.50,2.50,2
2,David,EQ3,3.50,2.50,2
3,David,EQ4,2.00,4.00,2
4,David,EQ5,3.00,3.50,2
5,Parker,EQ1,3.50,2.50,2
6,Parker,EQ2,4.00,2.00,2
7,Parker,EQ3,4.00,2.00,2
8,Parker,EQ4,1.00,4.00,1
9,Parker,EQ5,2.00,4.00,1


## Mean Effect (Baseline EQ1)

In [13]:
# Baseline effect from group first-round Q1 mean (baseline EQ1 Flat)
baseline_row = first_group_eq[first_group_eq["EQ"] == "EQ1"]
if baseline_row.empty:
    raise ValueError("Baseline EQ3 not found in first-round group summary")

baseline_q1 = baseline_row["First_Q1_Mean"].iloc[0]

effect_table = first_group_eq.copy()
effect_table["First_Q1_Effect"] = effect_table["First_Q1_Mean"] - baseline_q1

# Q1-focused ranks
# Higher score gets better rank (1 = best)
effect_table["First_Q1_Rank"] = _rank_desc(effect_table["First_Q1_Mean"])
effect_table["First_Q1_Effect_Rank"] = _rank_desc(effect_table["First_Q1_Effect"])

effect_table = effect_table.sort_values("EQ").reset_index(drop=True)
effect_table


,EQ,First_Q1_Mean,First_Q2_Mean,N_Subjects,First_Q1_Effect,First_Q1_Rank,First_Q1_Effect_Rank
0,EQ1,3.00,3.50,4,0.00,3,3
1,EQ2,3.50,2.50,4,0.50,1,1
2,EQ3,3.50,2.25,4,0.50,1,1
3,EQ4,1.75,4.00,4,-1.25,5,5
4,EQ5,2.00,3.75,4,-1.00,4,4


## Ranking Statistics

In [14]:
# Ranking statistics from final ranking scores
rank_df = pd.DataFrame(ranking, index=EQ_LIST)

rank_summary = (
    pd.DataFrame({
        "EQ": rank_df.index,
        "Ranking_Mean": rank_df.mean(axis=1).values,
        "Ranking_SD": rank_df.std(axis=1).values
    })
    .sort_values("EQ")
    .reset_index(drop=True)
)

rank_summary["Ranking_Rank"] = _rank_asc(rank_summary["Ranking_Mean"])
rank_summary


,EQ,Ranking_Mean,Ranking_SD,Ranking_Rank
0,EQ1,2.666667,0.57735,3
1,EQ2,1.333333,0.57735,1
2,EQ3,2.000000,1.00000,2
3,EQ4,4.666667,0.57735,5
4,EQ5,4.333333,0.57735,4


## Emotion Model Score (Tunable Vote + Raw)


In [15]:
# Emotion model score/rank
# A) Rebuilt-vote from RAW window (tunable weights + tunable RAW window)
vote_rows = []
raw_rows = []
vote_first_rows = []
raw_first_rows = []

vote_weights = VOTE_ANALYSIS_CONFIG["weights"]
window_start, window_end = get_raw_window_slice(VOTE_ANALYSIS_CONFIG)

for subject, sessions in emotion_data.items():
    seen_eq = set()

    for s in sessions:
        eq = s["eq"]

        selected_raw = s["raw"][window_start:window_end]
        selected_scored = [vote_weights[r] for r in selected_raw if r in vote_weights]  # unknown filtered
        session_vote_score = float(np.mean(selected_scored)) if selected_scored else np.nan

        # original vote4 reference (legacy baseline)
        vote4_label = s["vote"][3]
        vote4_score = SCORE_MAP[vote4_label]

        vote_rows.append({
            "subject": subject,
            "EQ": eq,
            "rebuilt_vote_score": session_vote_score,
            "rebuilt_vote_window_n": len(selected_raw),
            "rebuilt_vote_valid_n": len(selected_scored),
            "vote4_label": vote4_label,
            "vote4_score": vote4_score,
        })

        for raw_label in s["raw"]:
            raw_rows.append({
                "subject": subject,
                "EQ": eq,
                "raw_label": raw_label,
                "raw_is_valid": raw_label in SCORE_MAP,
                "raw_score": SCORE_MAP[raw_label] if raw_label in SCORE_MAP else np.nan,
            })

        if eq not in seen_eq:
            seen_eq.add(eq)
            vote_first_rows.append({
                "subject": subject,
                "EQ": eq,
                "rebuilt_vote_score": session_vote_score,
                "rebuilt_vote_window_n": len(selected_raw),
                "rebuilt_vote_valid_n": len(selected_scored),
                "vote4_label": vote4_label,
                "vote4_score": vote4_score,
            })
            for raw_label in s["raw"]:
                raw_first_rows.append({
                    "subject": subject,
                    "EQ": eq,
                    "raw_label": raw_label,
                    "raw_is_valid": raw_label in SCORE_MAP,
                    "raw_score": SCORE_MAP[raw_label] if raw_label in SCORE_MAP else np.nan,
                })

vote_long = pd.DataFrame(vote_rows)
raw_long = pd.DataFrame(raw_rows)
vote_first_long = pd.DataFrame(vote_first_rows)
raw_first_long = pd.DataFrame(raw_first_rows)

vote_all_summary = (
    vote_long.groupby("EQ", as_index=False)
    .agg(
        vote_all_score=("rebuilt_vote_score", "sum"),
        vote_all_score_mean=("rebuilt_vote_score", "mean"),
        vote_all_n=("rebuilt_vote_score", "size"),
        vote_all_valid_n=("rebuilt_vote_valid_n", "sum"),
        vote4_all_score=("vote4_score", "sum"),
        vote4_all_score_mean=("vote4_score", "mean"),
    )
    .sort_values("EQ")
    .reset_index(drop=True)
)
vote_all_summary["vote_all_rank"] = _rank_desc(vote_all_summary["vote_all_score_mean"].fillna(-np.inf))
vote_all_summary["vote4_all_rank"] = _rank_desc(vote_all_summary["vote4_all_score_mean"].fillna(-np.inf))

vote_first_summary = (
    vote_first_long.groupby("EQ", as_index=False)
    .agg(
        vote_first_score=("rebuilt_vote_score", "sum"),
        vote_first_score_mean=("rebuilt_vote_score", "mean"),
        vote_first_n=("rebuilt_vote_score", "size"),
        vote_first_valid_n=("rebuilt_vote_valid_n", "sum"),
        vote4_first_score=("vote4_score", "sum"),
        vote4_first_score_mean=("vote4_score", "mean"),
    )
    .sort_values("EQ")
    .reset_index(drop=True)
)
vote_first_summary["vote_first_rank"] = _rank_desc(vote_first_summary["vote_first_score_mean"].fillna(-np.inf))
vote_first_summary["vote4_first_rank"] = _rank_desc(vote_first_summary["vote4_first_score_mean"].fillna(-np.inf))

# B) Raw-based: use all raw entries, filter unknown (do not score unknown)
raw_all_summary = (
    raw_long.groupby("EQ", as_index=False)
    .agg(
        raw_all_total_n=("raw_label", "size"),
        raw_all_valid_n=("raw_is_valid", "sum"),
    )
)
raw_all_summary["raw_all_unknown_n"] = raw_all_summary["raw_all_total_n"] - raw_all_summary["raw_all_valid_n"]
raw_all_summary["raw_all_coverage"] = raw_all_summary["raw_all_valid_n"] / raw_all_summary["raw_all_total_n"]

raw_all_valid = raw_long[raw_long["raw_is_valid"]].copy()
raw_all_scores = (
    raw_all_valid.groupby("EQ", as_index=False)
    .agg(
        raw_all_score=("raw_score", "sum"),
        raw_all_score_mean=("raw_score", "mean"),
    )
)
raw_all_summary = raw_all_summary.merge(raw_all_scores, on="EQ", how="left")
raw_all_summary["raw_all_adjusted_score_mean"] = raw_all_summary["raw_all_score_mean"] * raw_all_summary["raw_all_coverage"]
raw_all_summary["raw_all_rank"] = _rank_desc(raw_all_summary["raw_all_score_mean"].fillna(-np.inf))
raw_all_summary["raw_all_adjusted_rank"] = _rank_desc(raw_all_summary["raw_all_adjusted_score_mean"].fillna(-np.inf))
raw_all_summary = raw_all_summary.sort_values("EQ").reset_index(drop=True)

raw_first_summary = (
    raw_first_long.groupby("EQ", as_index=False)
    .agg(
        raw_first_total_n=("raw_label", "size"),
        raw_first_valid_n=("raw_is_valid", "sum"),
    )
)
raw_first_summary["raw_first_unknown_n"] = raw_first_summary["raw_first_total_n"] - raw_first_summary["raw_first_valid_n"]
raw_first_summary["raw_first_coverage"] = raw_first_summary["raw_first_valid_n"] / raw_first_summary["raw_first_total_n"]

raw_first_valid = raw_first_long[raw_first_long["raw_is_valid"]].copy()
raw_first_scores = (
    raw_first_valid.groupby("EQ", as_index=False)
    .agg(
        raw_first_score=("raw_score", "sum"),
        raw_first_score_mean=("raw_score", "mean"),
    )
)
raw_first_summary = raw_first_summary.merge(raw_first_scores, on="EQ", how="left")
raw_first_summary["raw_first_rank"] = _rank_desc(raw_first_summary["raw_first_score_mean"].fillna(-np.inf))
raw_first_summary = raw_first_summary.sort_values("EQ").reset_index(drop=True)

emotion_summary = (
    vote_all_summary
    .merge(vote_first_summary, on="EQ", how="outer")
    .merge(raw_all_summary, on="EQ", how="outer")
    .merge(raw_first_summary, on="EQ", how="outer")
    .sort_values("EQ")
    .reset_index(drop=True)
)

print(f"Rebuilt vote config from RAW: weights={vote_weights}, raw_window=[{window_start}:{window_end})")
emotion_summary


Rebuilt vote config from RAW: weights={'positive': -1.0, 'neutral': 0.5, 'negative': 0.0}, raw_window=[0:4)


,EQ,vote_all_score,vote_all_score_mean,vote_all_n,vote_all_valid_n,vote4_all_score,vote4_all_score_mean,vote_all_rank,vote4_all_rank,vote_first_score,...,raw_all_adjusted_score_mean,raw_all_rank,raw_all_adjusted_rank,raw_first_total_n,raw_first_valid_n,raw_first_unknown_n,raw_first_coverage,raw_first_score,raw_first_score_mean,raw_first_rank
0,EQ1,-0.250,-0.027778,9,35,-3,-0.333333,5,2,-1.375,...,0.333333,1,1,16,16,0,1.0000,9.0,0.562500,1
1,EQ2,2.250,0.225000,10,39,-6,-0.600000,1,4,0.375,...,0.175000,5,5,16,15,1,0.9375,4.0,0.266667,4
2,EQ3,1.250,0.125000,10,39,-4,-0.400000,2,3,0.125,...,0.225000,4,4,16,16,0,1.0000,5.0,0.312500,3
3,EQ4,0.250,0.050000,5,20,-1,-0.200000,4,1,-0.250,...,0.300000,2,2,16,16,0,1.0000,6.0,0.375000,2
4,EQ5,0.625,0.125000,5,20,-3,-0.600000,2,4,0.500,...,0.250000,3,3,16,16,0,1.0000,4.0,0.250000,5


## Final Comparison Tables (Rank-Centric)


In [16]:
# 最終比較表（中文命名）

問券表 = (
    effect_table[["EQ", "First_Q1_Mean", "First_Q1_Effect", "First_Q1_Rank", "First_Q1_Effect_Rank"]]
    .merge(rank_summary[["EQ", "Ranking_Mean", "Ranking_Rank"]], on="EQ", how="left")
    .rename(columns={
        "First_Q1_Mean": "問券_Q1平均",
        "First_Q1_Effect": "問券_Q1效果值",
        "First_Q1_Rank": "問券_Q1排名",
        "First_Q1_Effect_Rank": "問券_Q1效果值排名",
        "Ranking_Mean": "問券_最終排序平均分",
        "Ranking_Rank": "問券_最終排序排名",
    })
    .sort_values("EQ")
    .reset_index(drop=True)
)

投票v1_全資料表 = (
    emotion_summary[[
        "EQ", "vote_all_score", "vote_all_score_mean", "vote_all_n", "vote_all_valid_n", "vote_all_rank",
        "vote4_all_score", "vote4_all_score_mean", "vote4_all_rank",
    ]]
    .rename(columns={
        "vote_all_score": "投票v1_全資料_Score",
        "vote_all_score_mean": "投票v1_全資料_Score Mean",
        "vote_all_n": "投票v1_全資料_資料數",
        "vote_all_valid_n": "投票v1_全資料_有效資料數",
        "vote_all_rank": "投票v1_全資料_排名",
        "vote4_all_score": "投票v0_全資料_Score",
        "vote4_all_score_mean": "投票v0_全資料_Score Mean",
        "vote4_all_rank": "投票v0_全資料_排名",
    })
    .merge(問券表[["EQ", "問券_Q1排名", "問券_最終排序排名"]], on="EQ", how="left")
    .sort_values("EQ")
    .reset_index(drop=True)
)

投票v1_首輪表 = (
    emotion_summary[[
        "EQ", "vote_first_score", "vote_first_score_mean", "vote_first_n", "vote_first_valid_n", "vote_first_rank",
        "vote4_first_score", "vote4_first_score_mean", "vote4_first_rank",
    ]]
    .rename(columns={
        "vote_first_score": "投票v1_首輪_Score",
        "vote_first_score_mean": "投票v1_首輪_Score Mean",
        "vote_first_n": "投票v1_首輪_資料數",
        "vote_first_valid_n": "投票v1_首輪_有效資料數",
        "vote_first_rank": "投票v1_首輪_排名",
        "vote4_first_score": "投票v0_首輪_Score",
        "vote4_first_score_mean": "投票v0_首輪_Score Mean",
        "vote4_first_rank": "投票v0_首輪_排名",
    })
    .merge(問券表[["EQ", "問券_Q1排名", "問券_最終排序排名"]], on="EQ", how="left")
    .sort_values("EQ")
    .reset_index(drop=True)
)

Raw_全資料表 = (
    emotion_summary[[
        "EQ",
        "raw_all_score", "raw_all_score_mean", "raw_all_rank",
        "raw_all_valid_n", "raw_all_unknown_n", "raw_all_coverage",
    ]]
    .rename(columns={
        "raw_all_score": "Raw_全資料_Score",
        "raw_all_score_mean": "Raw_全資料_Score Mean",
        "raw_all_rank": "Raw_全資料_排名",
        "raw_all_valid_n": "Raw_全資料_有效資料數",
        "raw_all_unknown_n": "Raw_全資料_unknown資料數",
        "raw_all_coverage": "Raw_全資料_覆蓋率",
    })
    .merge(問券表[["EQ", "問券_Q1排名", "問券_最終排序排名"]], on="EQ", how="left")
    .sort_values("EQ")
    .reset_index(drop=True)
)

Raw_首輪表 = (
    emotion_summary[[
        "EQ",
        "raw_first_score", "raw_first_score_mean", "raw_first_rank",
        "raw_first_valid_n", "raw_first_unknown_n", "raw_first_coverage",
    ]]
    .rename(columns={
        "raw_first_score": "Raw_首輪_Score",
        "raw_first_score_mean": "Raw_首輪_Score Mean",
        "raw_first_rank": "Raw_首輪_排名",
        "raw_first_valid_n": "Raw_首輪_有效資料數",
        "raw_first_unknown_n": "Raw_首輪_unknown資料數",
        "raw_first_coverage": "Raw_首輪_覆蓋率",
    })
    .merge(問券表[["EQ", "問券_Q1排名", "問券_最終排序排名"]], on="EQ", how="left")
    .sort_values("EQ")
    .reset_index(drop=True)
)

排名總覽表 = (
    問券表[["EQ", "問券_Q1排名", "問券_最終排序排名"]]
    .merge(投票v1_全資料表[["EQ", "投票v1_全資料_排名", "投票v0_全資料_排名"]], on="EQ", how="left")
    .merge(投票v1_首輪表[["EQ", "投票v1_首輪_排名", "投票v0_首輪_排名"]], on="EQ", how="left")
    .merge(Raw_全資料表[["EQ", "Raw_全資料_排名"]], on="EQ", how="left")
    .merge(Raw_首輪表[["EQ", "Raw_首輪_排名"]], on="EQ", how="left")
    .sort_values("EQ")
    .reset_index(drop=True)
)


def evaluate_vote_setting(weights, window_start, window_end):
    rows = []
    for subject, sessions in emotion_data.items():
        for s in sessions:
            eq = s["eq"]
            sel_raw = s["raw"][window_start:window_end]
            scored = [weights[r] for r in sel_raw if r in weights]
            score = float(np.mean(scored)) if scored else np.nan
            rows.append({"EQ": eq, "score": score})

    df = pd.DataFrame(rows)
    agg = df.groupby("EQ", as_index=False).agg(model_vote_score_mean=("score", "mean"))
    merged = agg.merge(問券表[["EQ", "問券_Q1平均", "問券_Q1排名"]], on="EQ", how="inner")

    corr_q1_mean = merged["model_vote_score_mean"].corr(merged["問券_Q1平均"], method="spearman")
    model_rank = merged["model_vote_score_mean"].rank(method="min", ascending=False)
    corr_q1_rank = model_rank.corr(merged["問券_Q1排名"], method="spearman")

    return corr_q1_mean, corr_q1_rank


tuning_rows = []
weight_candidates = [
    {"positive": 1.0, "neutral": 0.5, "negative": 0.0},
    {"positive": 1.0, "neutral": 0, "negative": -1.0},
    {"positive": -1.0, "neutral": 0, "negative": 1.0},
    {"positive": 0.0, "neutral": -0.5, "negative": 1.0},
    {"positive": 1.0, "neutral": -0.5, "negative": 0.0},
    {"positive": 0.0, "neutral": 1, "negative": 0.0},
    {"positive": -1.0, "neutral": 1.0, "negative": 0.0}
]
window_candidates = [(0, 4), (1, 4), (2, 4), (3, 4),(0, 1),(0, 2),(0, 3)]

for w in weight_candidates:
    for ws, we in window_candidates:
        c_mean, c_rank = evaluate_vote_setting(w, ws, we)
        tuning_rows.append({
            "投票v1_權重(作用於Raw)": str(w),
            "Raw_window": f"[{ws}:{we})",
            "與問券_Q1平均_Spearman": c_mean,
            "與問券_Q1排名_Spearman": c_rank,
        })

tuning_table = pd.DataFrame(tuning_rows).sort_values("與問券_Q1平均_Spearman", ascending=False).reset_index(drop=True)

print("問券表")
display(問券表)
print("投票v1_全資料表（含投票v0對照）")
display(投票v1_全資料表)
print("投票v1_首輪表（含投票v0對照）")
display(投票v1_首輪表)
print("Raw_全資料表")
display(Raw_全資料表)
print("Raw_首輪表")
display(Raw_首輪表)
print("排名總覽表")
display(排名總覽表)
print("投票v1調參表（權重+Raw window）")
display(tuning_table)


# 每位受測者比較表：問券Q1 mean / Raw mean / 投票v0 mean / 投票v1 mean
# 先把 subject key 正規化成小寫，避免 Parker/parker 這種大小寫不一致
questionnaire_norm = {str(k).strip().lower(): v for k, v in questionnaire.items()}
emotion_norm = {str(k).strip().lower(): v for k, v in emotion_data.items()}
subject_keys = sorted(set(questionnaire_norm.keys()) & set(emotion_norm.keys()))


def build_subject_compare_table(subject_key):
    q_entries = questionnaire_norm[subject_key]
    e_sessions = emotion_norm[subject_key]

    # 問券 Q1 mean (每個 EQ)
    q_rows = []
    for e in q_entries:
        q_rows.append({"EQ": e["eq"], "Q1": e["Q1"]})
    q_df = pd.DataFrame(q_rows)
    q_mean = q_df.groupby("EQ", as_index=False).agg(問券_Q1_mean=("Q1", "mean"))

    # Raw mean (每個 EQ, unknown 過濾)
    raw_rows = []
    for s in e_sessions:
        eq = s["eq"]
        for raw_label in s["raw"]:
            if raw_label in SCORE_MAP:
                raw_rows.append({"EQ": eq, "Raw_score": SCORE_MAP[raw_label]})
    raw_df = pd.DataFrame(raw_rows)
    raw_mean = raw_df.groupby("EQ", as_index=False).agg(Raw_mean=("Raw_score", "mean"))

    # 投票v0 mean (每個 EQ, 使用原始 vote 第4筆)
    v0_rows = []
    for s in e_sessions:
        eq = s["eq"]
        v0_rows.append({"EQ": eq, "投票v0_score": SCORE_MAP[s["vote"][3]]})
    v0_df = pd.DataFrame(v0_rows)
    v0_mean = v0_df.groupby("EQ", as_index=False).agg(投票v0_mean=("投票v0_score", "mean"))

    # 投票v1 mean (每個 EQ, 使用 raw window + weights)
    ws, we = get_raw_window_slice(VOTE_ANALYSIS_CONFIG)
    w = VOTE_ANALYSIS_CONFIG["weights"]
    v1_rows = []
    for s in e_sessions:
        eq = s["eq"]
        sel_raw = s["raw"][ws:we]
        scored = [w[r] for r in sel_raw if r in w]
        score = float(np.mean(scored)) if scored else np.nan
        v1_rows.append({"EQ": eq, "投票v1_score": score})
    v1_df = pd.DataFrame(v1_rows)
    v1_mean = v1_df.groupby("EQ", as_index=False).agg(投票v1_mean=("投票v1_score", "mean"))

    tbl = (
        q_mean
        .merge(raw_mean, on="EQ", how="outer")
        .merge(v0_mean, on="EQ", how="outer")
        .merge(v1_mean, on="EQ", how="outer")
        .sort_values("EQ")
        .reset_index(drop=True)
    )
    return tbl


subject_compare_tables = {k: build_subject_compare_table(k) for k in subject_keys}

# ALL: 所有受測者合併
all_q_rows = []
all_raw_rows = []
all_v0_rows = []
all_v1_rows = []
ws, we = get_raw_window_slice(VOTE_ANALYSIS_CONFIG)
w = VOTE_ANALYSIS_CONFIG["weights"]

for sk in subject_keys:
    for e in questionnaire_norm[sk]:
        all_q_rows.append({"EQ": e["eq"], "Q1": e["Q1"]})

    for s in emotion_norm[sk]:
        eq = s["eq"]
        for raw_label in s["raw"]:
            if raw_label in SCORE_MAP:
                all_raw_rows.append({"EQ": eq, "Raw_score": SCORE_MAP[raw_label]})

        all_v0_rows.append({"EQ": eq, "投票v0_score": SCORE_MAP[s["vote"][3]]})

        sel_raw = s["raw"][ws:we]
        scored = [w[r] for r in sel_raw if r in w]
        score = float(np.mean(scored)) if scored else np.nan
        all_v1_rows.append({"EQ": eq, "投票v1_score": score})

ALL比較表 = (
    pd.DataFrame(all_q_rows).groupby("EQ", as_index=False).agg(問券_Q1_mean=("Q1", "mean"))
    .merge(pd.DataFrame(all_raw_rows).groupby("EQ", as_index=False).agg(Raw_mean=("Raw_score", "mean")), on="EQ", how="outer")
    .merge(pd.DataFrame(all_v0_rows).groupby("EQ", as_index=False).agg(投票v0_mean=("投票v0_score", "mean")), on="EQ", how="outer")
    .merge(pd.DataFrame(all_v1_rows).groupby("EQ", as_index=False).agg(投票v1_mean=("投票v1_score", "mean")), on="EQ", how="outer")
    .sort_values("EQ")
    .reset_index(drop=True)
)

print("每位受測者比較表")
for sk in subject_keys:
    print(f"subject: {sk}")
    display(subject_compare_tables[sk])

print("subject: ALL")
display(ALL比較表)


問券表


,EQ,問券_Q1平均,問券_Q1效果值,問券_Q1排名,問券_Q1效果值排名,問券_最終排序平均分,問券_最終排序排名
0,EQ1,3.00,0.00,3,3,2.666667,3
1,EQ2,3.50,0.50,1,1,1.333333,1
2,EQ3,3.50,0.50,1,1,2.000000,2
3,EQ4,1.75,-1.25,5,5,4.666667,5
4,EQ5,2.00,-1.00,4,4,4.333333,4


投票v1_全資料表（含投票v0對照）


,EQ,投票v1_全資料_Score,投票v1_全資料_Score Mean,投票v1_全資料_資料數,投票v1_全資料_有效資料數,投票v1_全資料_排名,投票v0_全資料_Score,投票v0_全資料_Score Mean,投票v0_全資料_排名,問券_Q1排名,問券_最終排序排名
0,EQ1,-0.250,-0.027778,9,35,5,-3,-0.333333,2,3,3
1,EQ2,2.250,0.225000,10,39,1,-6,-0.600000,4,1,1
2,EQ3,1.250,0.125000,10,39,2,-4,-0.400000,3,1,2
3,EQ4,0.250,0.050000,5,20,4,-1,-0.200000,1,5,5
4,EQ5,0.625,0.125000,5,20,2,-3,-0.600000,4,4,4


投票v1_首輪表（含投票v0對照）


...

,EQ,問券_Q1_mean,Raw_mean,投票v0_mean,投票v1_mean
0,EQ1,3.5,0.857143,1.0,-0.750
1,EQ2,4.0,0.428571,0.0,-0.125
2,EQ3,4.0,0.714286,1.0,-0.625
3,EQ4,1.0,0.750000,1.0,-0.625
4,EQ5,2.0,0.250000,-1.0,0.125


subject: roger


,EQ,問券_Q1_mean,Raw_mean,投票v0_mean,投票v1_mean
0,EQ1,3.00,0.125,-1.0,0.3125
1,EQ2,3.75,0.000,-1.0,0.5000
2,EQ3,4.00,0.100,-1.0,0.3500
3,EQ4,2.00,0.250,-1.0,0.1250
4,EQ5,2.00,0.250,-1.0,0.1250


subject: ALL


,EQ,問券_Q1_mean,Raw_mean,投票v0_mean,投票v1_mean
0,EQ1,3.222222,0.342857,-0.333333,-0.027778
1,EQ2,3.700000,0.179487,-0.600000,0.225000
2,EQ3,3.777778,0.230769,-0.400000,0.125000
3,EQ4,1.800000,0.300000,-0.200000,0.050000
4,EQ5,2.200000,0.250000,-0.600000,0.125000
